# 01 · Exploratory data analysis

**Purpose.** Understand the two datasets before designing the matching pipeline:

* `Current Portfolio.xlsx`: the existing estate (unit level)
* `Marketed Warehouses.xlsx`: new listings to compare against it

The brief warns of *missing and inaccurate values*, so we go through the variables **one at a time**,
starting with those that should be numeric, and note what each issue means for similarity matching.
The notebook ends with the cleaning decisions to carry into `src/`.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Store each chart inside the notebook, so it shows wherever the notebook is opened.
# JupyterLab draws them itself; the classic notebook loads plotly.js from a CDN.
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Resolve the repo root whether the notebook is run from notebooks/ or the root.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"

Both files are loaded **raw** (`dtype=object`), so every cell keeps exactly the value Excel holds
and nothing is converted before we've looked at it.

In [2]:
raw_portfolio = pd.read_excel(DATA / "Current Portfolio.xlsx", dtype=object)
raw_marketed = pd.read_excel(DATA / "Marketed Warehouses.xlsx", dtype=object)

print(f"Portfolio: {raw_portfolio.shape[0]:,} rows x {raw_portfolio.shape[1]} cols")
print(f"Marketed:  {raw_marketed.shape[0]:,} rows x {raw_marketed.shape[1]} cols")
raw_portfolio.head()

Portfolio: 1,250 rows x 14 cols
Marketed:  5 rows x 14 cols


,Property Id,Industrial Estate Name,Unit Name,Region,Latitude,Longitude,Car Parking Spaces #,Size sqm,Build Year,Yard Depth m,Min. Eaves m,Max. Eaves m,Doors #,EPC Rating
0,1,Ivory Wharf 66,Unit 1,Midlands,51.92727,-0.987484,22,7508.729478,1999,NaN,4,5.2,0,NaN
1,2,Ivory Wharf 66,Unit 2,Midlands,51.928783,-0.988291,22,13612.939196,1975,NaN,4,5.2,0,NaN
2,3,Ivory Wharf 66,Unit 3,Midlands,51.928141,-0.987723,22,4065.156838,1983,NaN,4,5.2,0,NaN
3,4,Ivory Wharf 66,Unit 4,Midlands,51.927537,-0.987248,22,1421.273392,1998,NaN,4,5.2,0,NaN
4,5,Ivory Wharf 66,Unit 5,Midlands,51.927563,-0.987496,22,2857.252395,1993,NaN,4,5.2,0,NaN


In [3]:
raw_marketed

,Property Id,Industrial Estate Name,Unit Name,Region,Latitude,Longitude,Car Parking Spaces #,Size sqm,Build Year,Yard Depth m,Min. Eaves m,Max. Eaves m,Doors #,EPC Rating
0,NaN,Cherry Lane,Unit 10,North West,52.467186,-3.784003,30,16860.130016,1977,NaN,NaN,NaN,NaN,NaN
1,NaN,Tech Hub,Unit 8,South East,NaN,NaN,23,905.523948,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Spitfire Park,Unit 42,Midlands,53.505831,-1.905054,49,2294.49811,1976,NaN,NaN,NaN,NaN,B
3,NaN,Stable Lane,Unit 14,South East,52.078733,-0.349448,48,617.297521,2000,NaN,7,7,16,C
4,NaN,Chancery Depot,Unit 2,NaN,52.485703,-4.478974,22,16613.786473,1966,NaN,5.21,5.46,3,A


The schema is the same between the two notebooks, but the market warehouse data doesn't have property_id. There are signs of missing data which we will investigate further

**What a row is.** Each row of the portfolio is one **unit** (a lettable building or part of one), identified
by `Property Id`. Units belong to an **industrial estate** (`Industrial Estate Name`), and an estate can hold
one unit or many. This matters later: some values turn out to be recorded per estate and copied onto every unit,
and an estate with many units could crowd a "most similar" list.

In [4]:
units_per_estate = raw_portfolio.groupby("Industrial Estate Name").size()
print(f"{len(raw_portfolio):,} units in {len(units_per_estate)} estates "
      f"(Property Id unique: {raw_portfolio['Property Id'].is_unique})")

bands = pd.cut(units_per_estate, [0, 1, 5, 10, 20, np.inf], labels=["1", "2-5", "6-10", "11-20", "21+"])
pd.DataFrame({
    "estates": units_per_estate.groupby(bands, observed=False).size(),
    "units": units_per_estate.groupby(bands, observed=False).sum(),
}).assign(**{"% of units": lambda t: (t["units"] / t["units"].sum() * 100).round(0)}).rename_axis("units in estate")

1,250 units in 263 estates (Property Id unique: True)


,estates,units,% of units
units in estate,,,
1,125,125,10.0
2-5,73,232,19.0
6-10,39,312,25.0
11-20,16,223,18.0
21+,10,358,29.0


* **1,250 units in 263 estates**, and `Property Id` is unique, so it identifies a unit.
* **125 estates (48%) have a single unit**, but they hold only 10% of the units. The other 138 estates are
  multi-unit (median 5 units each), and they hold 90% of the units.
* **The large estates dominate the unit count:** the 10 estates with 21+ units hold 29% of all units. The largest
  is Quartz Wharf 16 with 64 units, then Ruby Arc 96 with 48.
* So a statistic counted per unit is weighted heavily towards the big estates. Where an estate-level value is
  copied onto its units, count per estate as well (as for car parking in 2.4).

## 2. Numeric variables, one by one

`profile(col)` gives the same view of every column:

1. **blank** cells
2. **non-numeric** values (placeholder text like `X` or `TBC`), with counts
3. **zeros**, which are often a "not recorded" placeholder in disguise
4. a summary and histogram of the values that *are* numbers

In [5]:
def profile(col: str) -> None:
    """Print blanks / non-numeric values / zeros for one portfolio column, then summarise and plot it.

    Args:
        col: portfolio column name.
    """
    raw = raw_portfolio[col]
    num = pd.to_numeric(raw, errors="coerce")  # non-numbers become NaN, for this view only
    non_numeric = raw[raw.notna() & num.isna()]

    print(f"{col}  ({len(raw):,} units)")
    print(f"  blank:       {raw.isna().sum():>5}")
    print(f"  non-numeric: {len(non_numeric):>5}  {non_numeric.astype(str).value_counts().to_dict()}")
    print(f"  zero:        {(num == 0).sum():>5}")
    print(f"  numeric:     {num.notna().sum():>5}")
    display(num.describe().round(2).to_frame().T)

    px.histogram(x=num, nbins=60, title=col, labels={"x": col}, height=320).show()

### 2.1 Size (`Size sqm` column)

In [6]:
profile("Size sqm")

Size sqm  (1,250 units)
  blank:           0
  non-numeric:     0  {}
  zero:            0
  numeric:      1250


,count,mean,std,min,25%,50%,75%,max
Size sqm,1250.0,24472.88,77986.01,261.69,2122.71,4692.57,12872.76,1195450.57


* **Complete:** no blanks, no placeholder text, no zeros.
* **Very skewed:** 262 to 1.2m (median ~4,700). Most units are small, with a long tail of very large ones.

**Are these values realistic for Indurent?** Indurent's website (checked November 2026; 10.764 sq ft = 1 sqm):

| Figure | Where on indurent.com | sq ft | ≈ sqm |
|---|---|---|---|
| Total portfolio | Site navigation menu | "c. 30m sq ft of assets" | ~2.8m |
| Total logistics space | [Logistics page](https://www.indurent.com/logistics/) | "28 million square feet" | ~2.6m |
| Total portfolio | [About page](https://www.indurent.com/about-indurent/) | "40m sq ft" | ~3.7m |
| Small units (general definition) | [Small units page](https://www.indurent.com/small-industrial-units-to-rent/) | 500–5,000 | 46–465 |
| Largest units named on the site | [Homepage](https://www.indurent.com/) case studies / news | 129,000 and 167,000 | 12,000–15,500 |

Indurent doesn't publish a maximum unit size. Its homepage search tops out at a "100,000+ ft²" band.

In [7]:
SQFT_PER_SQM = 10.764
size = pd.to_numeric(raw_portfolio["Size sqm"])
total = size.sum()
print(f"Column total: {total / 1e6:.1f}m")
print(f"  if the values are sqm:  {total * SQFT_PER_SQM / 1e6:.0f}m sq ft")
print(f"  if the values are sq ft: {total / 1e6:.1f}m sq ft   (Indurent states 28-40m sq ft)")

Column total: 30.6m
  if the values are sqm:  329m sq ft
  if the values are sq ft: 30.6m sq ft   (Indurent states 28-40m sq ft)


**Rescaling only the largest units?** We first considered treating the units above Indurent's
largest plausible size as individual input errors (sq ft typed into a sqm field) and dividing just those
by 10.764. That doesn't hold up:

* it creates a cliff at whatever threshold is chosen, reversing the order of units either side of it;
* it leaves the column total at ~12m sqm, still about 4x Indurent's stated portfolio.

The **whole column** matches Indurent's stated 28–40m sq ft, so the more likely explanation is that
**`Size sqm` is actually in square feet**. Two independent checks below support this.

In [8]:
# Read as sq ft: how do units fall into the size bands Indurent's own website search uses?
# Max eaves height is recorded separately in metres; zeros mean "not recorded" (see 2.5).
size_band = pd.cut(size, [0, 5_000, 10_000, 25_000, 50_000, 100_000, np.inf],
                   labels=["0-5k", "5-10k", "10-25k", "25-50k", "50-100k", "100k+"])
max_eaves = pd.to_numeric(raw_portfolio["Max. Eaves m"], errors="coerce")
pd.DataFrame({
    "units": size_band.value_counts().sort_index(),
    "median max eaves (m)": max_eaves.where(max_eaves > 0).groupby(size_band).median(),
})

,units,median max eaves (m)
Size sqm,,
0-5k,650,6.6
5-10k,228,6.5
10-25k,164,6.7
25-50k,92,9.0
50-100k,52,12.0
100k+,64,12.5


In [9]:
# Estate totals: multi-let estates are typically tens to a few hundred thousand sq ft.
by_estate = size.groupby(raw_portfolio["Industrial Estate Name"]).agg(["count", "sum"])
multi_let = by_estate.loc[by_estate["count"] > 1, "sum"]
print(f"Multi-unit estates ({len(multi_let)}): total size median {multi_let.median():,.0f}, "
      f"10th-90th percentile {multi_let.quantile(0.1):,.0f} - {multi_let.quantile(0.9):,.0f}")
print(f"Single-unit estates ({(by_estate['count'] == 1).sum()}): median {by_estate.loc[by_estate['count'] == 1, 'sum'].median():,.0f}")
print(f"Units under 500 (below Indurent's small-unit definition): {(size < 500).sum()}")
print(f"Units over 1m: {(size > 1_000_000).sum()}")

Multi-unit estates (138): total size median 59,659, 10th-90th percentile 13,763 - 283,549
Single-unit estates (125): median 57,320
Units under 500 (below Indurent's small-unit definition): 10
Units over 1m: 1


**Read as sq ft, the distribution makes sense:**

* **Eaves height rises with size, as it should.** Units under 25,000 sq ft have a median max eaves of
  ~6.5 m (typical small/mid MLI), and units over 50,000 sq ft have ~12 m (typical big-box logistics).
  Read as sqm, a 25,000–50,000 sqm (270k–540k sq ft) shed with 9 m eaves would be unusually low.
* **Estate sizes look like MLI.** The median multi-unit estate is ~60,000 sq ft (10th–90th percentile
  ~14,000–284,000). Read as sqm, the median estate would be ~645,000 sq ft, very large for multi-let.
* **The median unit (~4,700 sq ft) matches Indurent's small-unit offer** (500–5,000 sq ft), and 650 units
  fall in the website's smallest "0–5k" band.
* **The single-unit estates (median ~57,000 sq ft) look like mid-box / big-box sheds,** consistent with
  Indurent's logistics portfolio.

**Outliers or believable?** Read as sq ft, the tails are believable rather than errors:

* **Top end:** the largest unit is ~1.2m sq ft (~111,000 sqm), the only one over 1m. That's at the top of
  the UK big-box market but real buildings of this size exist. These units are large compared with the
  sizes Indurent names on its website, so they're worth a note, not a correction.
* **Bottom end:** 10 units are under 500 sq ft (smallest 262 sq ft ≈ 24 sqm). That's below Indurent's
  small-unit definition, more like a lock-up or storage unit. It's possible, but a mild flag.

**Decision:** the values are **square feet**, so rename the column to `Size sq ft` in section 3 and leave the
values unchanged. Sq ft is the unit UK industrial property is quoted in, and the one Indurent uses. Apply the
same to the marketed file, and note the assumption in the write-up. Flag units under 500 sq ft for a light
check, and keep the big-box tail.

### 2.2 Latitude and Longitude

In [10]:
profile("Latitude")

Latitude  (1,250 units)
  blank:          80
  non-numeric:     0  {}
  zero:            0
  numeric:      1170


,count,mean,std,min,25%,50%,75%,max
Latitude,1170.0,52.29,1.3,49.41,51.26,52.23,53.15,57.08


In [11]:
profile("Longitude")

Longitude  (1,250 units)
  blank:          80
  non-numeric:     0  {}
  zero:            0
  numeric:      1170


,count,mean,std,min,25%,50%,75%,max
Longitude,1170.0,-1.44,1.54,-5.55,-2.49,-1.59,-0.17,2.15


In [12]:
# Units with missing coordinates: could we borrow them from another unit in the same estate?
lat = pd.to_numeric(raw_portfolio["Latitude"])
lon = pd.to_numeric(raw_portfolio["Longitude"])
missing = lat.isna()
estates_with_coords = raw_portfolio.loc[lat.notna(), "Industrial Estate Name"].unique()
recoverable = raw_portfolio.loc[missing, "Industrial Estate Name"].isin(estates_with_coords)
print(f"{missing.sum()} units missing coordinates; {recoverable.sum()} could borrow from a sibling unit")

80 units missing coordinates; 1 could borrow from a sibling unit


* **80 units have no coordinates.** 79 of them are in estates where *no* unit has coordinates, so there's
  nothing to borrow. Their location will be unknown.

In [13]:
# Map with coastlines, so points in the sea are obvious. scatter_geo draws Plotly's built-in
# coastline as SVG: no tile server or WebGL, so it works in any renderer.
points = raw_portfolio.assign(Latitude=lat, Longitude=lon).dropna(subset=["Latitude"])
fig = px.scatter_geo(points, lat="Latitude", lon="Longitude", color="Region",
                     hover_name="Industrial Estate Name", hover_data=["Unit Name"],
                     height=800, title="Portfolio units coloured by labelled region")
fig.update_geos(resolution=50, showland=True, landcolor="#f2f2f2", showcountries=True,
                lataxis_range=[49, 59.5], lonaxis_range=[-8.5, 3])
fig.update_traces(marker={"size": 5})
fig.show()

* **Region labels often don't match where the points are** (the colours on the map). Checked against region boundaries below.
* Many points are visibly in the sea, mostly in the English Channel. The coastline check below counts them.

**Checking against a real coastline.** The map is a visual check. To count the points in the sea properly,
test each one against the UK's land boundary from [Natural Earth](https://www.naturalearthdata.com/)
(1:10m admin-0 countries, public domain). Distances are measured in metres on the British National
Grid (EPSG:27700).

**Why allow 1 km offshore.** Natural Earth's 1:10m coastline is drawn for country-scale maps: its points are
often hundreds of metres apart, so it cuts corners and smooths over estuaries, docks, harbours and reclaimed
land. Industrial estates are often in exactly those places (ports, docksides, estuaries, reclaimed marsh), so a
correctly recorded unit at the water's edge can land a few hundred metres "offshore" on the simplified line.
Points under 1 km out are therefore treated as coastal rather than wrong. The table below shows whether that
cutoff separates near-misses from genuinely wrong points.

In [14]:
import urllib.request

import geopandas as gpd

# Download the boundary file once into data/external/ (gitignored).
NE_COUNTRIES = DATA / "external" / "ne_10m_admin_0_countries.zip"
if not NE_COUNTRIES.exists():
    NE_COUNTRIES.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip", NE_COUNTRIES
    )

countries = gpd.read_file(NE_COUNTRIES)
uk_land = countries.loc[countries["ADMIN"] == "United Kingdom"].to_crs(27700).geometry.union_all()

located = gpd.GeoDataFrame(
    points, geometry=gpd.points_from_xy(points["Longitude"], points["Latitude"]), crs=4326
).to_crs(27700)
located["km_offshore"] = located.distance(uk_land) / 1000
located["land_check"] = pd.cut(located["km_offshore"], [-1, 0, 1, 5, np.inf],
                               labels=["on land", "coastal (< 1 km)", "offshore 1-5 km", "offshore 5+ km"])
located["land_check"].value_counts(sort=False).to_frame("units")

,units
land_check,
on land,868
coastal (< 1 km),24
offshore 1-5 km,8
offshore 5+ km,270


In [15]:
# Offshore units by estate (1 km+), furthest first.
offshore = located[located["km_offshore"] >= 1]
print(f"{len(offshore)} units in {offshore['Industrial Estate Name'].nunique()} estates are 1 km+ offshore "
      f"({len(offshore) / len(located):.0%} of units with coordinates)")
(offshore.groupby("Industrial Estate Name")
 .agg(units=("Unit Name", "size"), region=("Region", "first"),
      km_offshore=("km_offshore", "median"), lat=("Latitude", "first"), lon=("Longitude", "first"))
 .round({"km_offshore": 1, "lat": 3, "lon": 3})
 .sort_values("km_offshore", ascending=False))

278 units in 46 estates are 1 km+ offshore (24% of units with coordinates)


,units,region,km_offshore,lat,lon
Industrial Estate Name,,,,,
Stone Mews 79,1,South East,129.0,49.408,-1.977
Sapphire Square 28,1,North East,104.3,56.002,-0.140
Quartz Court 93,2,South East,103.6,49.841,-0.112
Zinc Loft 64,5,South East,102.1,50.363,2.149
Onyx Beacon 83,8,South East,97.4,50.270,1.904
Velvet Forge 5,1,South East,95.6,49.778,-0.789
Umber Square 75,3,South East,93.0,49.956,0.686
Granite Court 9,1,South East,87.8,49.969,0.030
Ivory Quay 58,1,South East,83.7,49.931,-0.640


* **278 units (24% of units with coordinates) in 46 estates are 1 km+
  offshore**, 270 of them 5 km+ out. They're mostly in the English Channel, with others in the North Sea,
  the Bristol Channel, the Irish Sea, and Scottish firths and bays.
* **24 units are within 1 km of the coast.** That's within the precision of the generalised coastline,
  so treat them as valid.
* **The 1 km cutoff sits in a clear gap.** Only 8 units are 1-5 km out, against 270 at 5 km+: the wrong points
  are mostly far out to sea, not just past the line. Widening the cutoff to 2 km would change 1 unit.
* **Tightening it would wrongly drop real estates.** At 500 m, 19 more units would count as offshore: almost all
  of Stone Mews 81 (14 units, 0.56-0.74 km out, north Norfolk coast) and Ruby Mill 13 (4 units, 0.51-0.66 km,
  on the Wash). Both are whole estates in plausible coastal spots, and the Wash is largely reclaimed land, where
  a generalised coastline is least accurate. The same 1 km allowance is used for region boundaries (below),
  which come from the same generalised source.

**Decision:** flag units **1 km+ offshore as invalid locations**. Their location score is unavailable, and
they're excluded from distance-based filters such as "within 10 miles of a major city". Location quality
is a real limitation of this dataset: about 30% of units have either no coordinates or invalid ones.

**Are units within an estate close together?** Units on one industrial estate should be within a few
hundred metres of each other. If most of an estate's units were together but a few were far away, those
few would be unit-level errors we could fix by borrowing the estate's location. Measure each unit's
distance from its estate's centre (the median of its units' coordinates).

In [16]:
multi_unit = located[located.groupby("Industrial Estate Name")["Unit Name"].transform("size") > 1].copy()
centres = multi_unit.groupby("Industrial Estate Name").geometry.apply(
    lambda g: gpd.points_from_xy([g.x.median()], [g.y.median()])[0]
)
multi_unit["m_from_centre"] = multi_unit.distance(
    gpd.GeoSeries(multi_unit["Industrial Estate Name"].map(centres), crs=multi_unit.crs)
)

print(f"{multi_unit['Industrial Estate Name'].nunique()} multi-unit estates with coordinates, {len(multi_unit)} units")
print(f"Distance from estate centre: median {multi_unit['m_from_centre'].median():.0f} m, "
      f"99th percentile {multi_unit['m_from_centre'].quantile(0.99):.0f} m, max {multi_unit['m_from_centre'].max():.0f} m")
print(f"Units more than 250 m from their estate centre: {(multi_unit['m_from_centre'] > 250).sum()}")

# Is any estate split between land and sea?
mixed = located.groupby("Industrial Estate Name")["land_check"].nunique()
print(f"Estates with units in more than one land-check category: {(mixed > 1).sum()}")

127 multi-unit estates with coordinates, 1100 units
Distance from estate centre: median 57 m, 99th percentile 138 m, max 156 m
Units more than 250 m from their estate centre: 0
Estates with units in more than one land-check category: 1


* **No unit-level outliers.** Every unit is within ~150 m of its estate's centre (median 57 m). That's
  realistic and suspiciously uniform, as if each estate were given a centre point with units scattered
  around it.
* **When an estate's location is wrong, it's wrong for every unit:** all 46 offshore estates are entirely
  offshore. The same holds for missing coordinates (79 of 80 are whole estates) and region labels (only 2
  estates mix regions).
* The one estate that spans two land-check categories, *Onyx Harbour 29*, just straddles the coastline:
  1 unit on land and 5 within 70 m of it, all valid.
* **So a bad location can't be repaired from sibling units.** Location quality has to be judged **per estate**:
  on land, offshore or missing.

**Do region labels match the coordinates?**

Test each **valid** location (on land or within 1 km of the coast, from the coastline check above) against the actual region
boundaries from Natural Earth (1:10m admin-1). England's regions map directly onto the dataset's labels,
except East and West Midlands, which the dataset merges into "Midlands".

In [17]:
NE_REGIONS = DATA / "external" / "ne_10m_admin_1_states_provinces.zip"
if not NE_REGIONS.exists():
    urllib.request.urlretrieve(
        "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip", NE_REGIONS
    )

ENGLAND_TO_LABEL = {"East": "East of England", "East Midlands": "Midlands", "West Midlands": "Midlands",
                    "Greater London": "London", "Yorkshire and the Humber": "Yorks. & Humber"}
areas = gpd.read_file(NE_REGIONS, where="adm0_a3 = 'GBR'")
areas["label"] = np.where(areas["geonunit"] == "England",
                          areas["region"].replace(ENGLAND_TO_LABEL), areas["geonunit"])
regions = areas.dissolve("label")[["geometry"]].reset_index().to_crs(27700)

valid = located[located["km_offshore"] < 1]
# Nearest region rather than strictly "inside", so coastal points just offshore still get a region.
actual = gpd.sjoin_nearest(valid, regions, how="left")["label"]
actual = actual[~actual.index.duplicated()]

print(f"Units inside their labelled region: {(valid['Region'] == actual).mean():.0%} "
      f"of {len(valid)} units with valid locations")
pd.crosstab(valid["Region"], actual, rownames=["labelled"], colnames=["actual region"], margins=True)

Units inside their labelled region: 26% of 892 units with valid locations


actual region,East of England,London,Midlands,North East,North West,Scotland,South East,South West,Wales,Yorks. & Humber,All
labelled,,,,,,,,,,,
East of England,10,0,1,0,0,0,0,0,0,0,11
London,61,0,23,0,0,0,22,0,0,0,106
Midlands,4,0,117,0,54,0,21,19,1,1,217
North East,0,0,0,2,2,0,0,0,0,11,15
North West,2,0,69,20,11,0,0,1,10,6,119
Scotland,0,0,0,0,4,8,0,0,0,0,12
South East,98,11,53,0,0,0,53,56,0,0,271
South West,0,0,9,0,0,0,0,24,66,0,99
Wales,0,0,0,0,0,0,0,7,5,0,12


In [18]:
# How far are the mismatched units from the region they're labelled with?
region_shape = regions.set_index("label").geometry
mismatched = valid[valid["Region"] != actual]
km_to_label = pd.Series(
    [point.distance(region_shape[label]) / 1000 for point, label in zip(mismatched.geometry, mismatched["Region"])],
    index=mismatched.index,
)
print(f"{len(mismatched)} mismatched units: median {km_to_label.median():.0f} km outside their labelled region, "
      f"{(km_to_label <= 25).mean():.0%} within 25 km of it, furthest {km_to_label.max():.0f} km")

estates = pd.DataFrame({"labelled": valid["Region"], "actual": actual}).groupby(valid["Industrial Estate Name"]).first()
print(f"Estates whose label matches their location: {(estates['labelled'] == estates['actual']).mean():.0%} "
      f"of {len(estates)}")

656 mismatched units: median 38 km outside their labelled region, 41% within 25 km of it, furthest 167 km
Estates whose label matches their location: 31% of 151


* **Only about a quarter of units (26%) are inside their labelled region**, and 31% of estates. Region
  labels are per estate, so this is an estate-level problem, like the coordinates.
* **No unit labelled "London" is in London.** Most are in the East of England, the Midlands or the South East.
  "South West" units are mostly in Wales, "Wales" units mostly in the South West, and "North East" units
  mostly in Yorkshire.
* **Some are near misses but many aren't.** The mismatched units are a median 38 km outside their labelled
  region (41% within 25 km, so border cases), and up to ~170 km. That's not a consistent shift in one
  direction, so we can't tell whether the label or the coordinates are wrong.

**Decision:** don't use `Region` as a feature. Where a region is needed (for reporting, or a prompt like
"listings in the Midlands"), **derive it from the coordinates** using these boundaries, for valid locations
only. Note in answers that the dataset's own region labels are unreliable.

**Could the coordinates simply have been shifted?** On the map, many points look displaced towards the
south-east. If the region labels are right and the coordinates were moved by one systematic transformation,
reversing it should put estates back in their labelled regions. Location errors are per estate (see above), so
use one centre point per estate.

First, a quick test. A shift or rotation moves every point by the same rule, so distances *between* estates
don't change. If the estates sharing a label are spread wider than their region, no shift or rotation can
fit them inside it.

In [19]:
located_km = located.assign(x=located.geometry.x / 1000, y=located.geometry.y / 1000)
estate_xy = located_km.groupby("Industrial Estate Name").agg(
    x=("x", "median"), y=("y", "median"), region=("Region", "first")
)
estate_xy.groupby("region").agg(
    estates=("x", "size"),
    east_west_km=("x", lambda v: v.max() - v.min()),
    north_south_km=("y", lambda v: v.max() - v.min()),
).round(0)

,estates,east_west_km,north_south_km
region,,,
East of England,2,38.0,79.0
London,20,217.0,299.0
Midlands,36,264.0,310.0
North East,5,161.0,229.0
North West,24,285.0,328.0
Scotland,9,214.0,238.0
South East,71,338.0,428.0
South West,13,206.0,265.0
Wales,3,57.0,238.0


In [20]:
import shapely
from scipy.optimize import minimize

shapes = region_shape.simplify(200)  # 200 m tolerance: speeds up distances, negligible at this scale
xy = estate_xy[["x", "y"]].to_numpy() * 1000
labels = estate_xy["region"].to_numpy()


def km_outside(points: np.ndarray) -> np.ndarray:
    """Distance (km) from each estate to its labelled region; 0 if inside."""
    out = np.empty(len(points))
    for label in np.unique(labels):
        idx = labels == label
        out[idx] = shapely.distance(shapes[label], shapely.points(points[idx])) / 1000
    return out


# 1) Best pure shift: grid search over +/-300 km in 10 km steps, maximising estates inside their region.
shifts = [(dx, dy) for dx in range(-300, 301, 10) for dy in range(-300, 301, 10)]
best_dx, best_dy = max(shifts, key=lambda s: np.mean(km_outside(xy + np.array(s) * 1000) == 0))

# 2) Best affine transform (scale, rotation, shear and shift) about the map centre,
#    minimising the mean distance outside the labelled region.
centre = xy.mean(axis=0)


def affine(t: np.ndarray, points: np.ndarray) -> np.ndarray:
    return (points - centre) @ np.array([[t[0], t[1]], [t[2], t[3]]]).T + centre + np.array(t[4:]) * 1000


fit = minimize(lambda t: km_outside(affine(t, xy)).mean(), [1, 0, 0, 1, best_dx, best_dy],
               method="Nelder-Mead", options={"maxiter": 4000})
scale = np.linalg.svd(fit.x[:4].reshape(2, 2), compute_uv=False)
print(f"Best shift: {best_dx:+} km east, {best_dy:+} km north")
print(f"Best affine fit scales the map by {scale.min():.2f}-{scale.max():.2f}x")

versions = {
    "as recorded": xy,
    "best shift": xy + np.array([best_dx, best_dy]) * 1000,
    "best affine fit": affine(fit.x, xy),
}
inside = pd.DataFrame({name: km_outside(points) == 0 for name, points in versions.items()}, index=estate_xy.index)
by_region = inside.groupby(labels).mean().mul(100).round(0)
by_region["estates"] = pd.Series(labels).value_counts()
by_region.loc["ALL ESTATES"] = [*inside.mean().mul(100).round(0), len(labels)]
by_region.astype(int).rename_axis("% of estates inside labelled region")

Best shift: -40 km east, +30 km north
Best affine fit scales the map by 0.26-0.73x


,as recorded,best shift,best affine fit,estates
% of estates inside labelled region,,,,
East of England,50,50,0,2
London,0,0,5,20
Midlands,39,50,69,36
North East,40,40,20,5
North West,8,29,42,24
Scotland,44,89,56,9
South East,17,23,59,71
South West,38,46,15,13
Wales,33,33,0,3


* **The spread test rules it out.** The 20 estates labelled "London" span ~217 km east–west and ~299 km
  north–south, but Greater London is only about 50 × 40 km. Every region's estates are spread far wider than
  the region itself, so no shift or rotation can fit them inside it.
* **Fitting anyway confirms it.** The best shift (~40 km west, ~30 km north) raises the share of estates inside
  their labelled region from 23% to only 32%, and still puts **no** London estate in London.
* **The affine fit's 45% is misleading.** It shrinks the map to roughly 0.3–0.7 of its size, pulling
  everything towards the Midlands and South East, the biggest groups. That helps those labels but makes
  others worse (Wales 33% → 0%, South West 38% → 15%, East of England 50% → 0%), and London is still only 1 of 20.
  It's the optimiser exploiting the largest groups, not a real correction.

**Conclusion:** the south-easterly look on the map isn't a systematic offset. Labels and coordinates disagree
independently, estate by estate, which fits the other signs that the data was generated or anonymised:
generated estate names, whole estates in the sea, and uniformly tight unit clusters. The decision above
stands: don't trust `Region`, and derive it from valid coordinates when needed.

**Location confidence.** Keeping only units where the coordinates *and* the region label agree would leave
under a fifth of the estate, with no London units at all. So instead of one cut, combine the checks above
into a confidence level per unit:

| Confidence | Rule |
|---|---|
| **high** | Inside its labelled region (with a 1 km allowance for boundary precision) |
| **medium** | On land (or within 1 km of the coast), but outside its labelled region, so we can't tell which is wrong |
| **unknown** | No coordinates, or 1 km+ offshore and outside its labelled region |

High doesn't need a separate coastline test: the region shapes are land, so a point within 1 km of its
region is already on land or right at the coast.

In [21]:
km_to_labelled = pd.Series(
    [point.distance(region_shape[label]) / 1000 for point, label in zip(located.geometry, located["Region"])],
    index=located.index,
)
on_land = located.index[located["km_offshore"] < 1]
agrees = located.index[km_to_labelled <= 1]  # region check only; no separate coastline test

location_confidence = pd.Series("unknown", index=raw_portfolio.index)
location_confidence[on_land] = "medium"
location_confidence[agrees] = "high"
location_confidence = pd.Categorical(location_confidence, categories=["high", "medium", "unknown"], ordered=True)

summary = pd.crosstab(raw_portfolio["Region"], location_confidence, margins=True, margins_name="All")
summary.columns.name = "location confidence"
summary.sort_values("All", ascending=False)

location confidence,high,medium,unknown,All
Region,,,,
All,240,652,358,1250
South East,57,214,107,378
Midlands,117,100,84,301
North West,11,108,46,165
London,0,106,38,144
South West,24,75,35,134
Yorks. & Humber,6,24,20,50
Scotland,8,4,26,38
North East,2,13,1,16


* **240 high (19%), 652 medium (52%), 358 unknown (29%).** About 71% of units have a usable location
  (high + medium).
* **High confidence is lopsided:** about half of it is the Midlands (117), and **no London unit** reaches it.
  A "high only" analysis would effectively describe the Midlands estates, so it's too narrow to be the
  default.

**How to use it:**
* Distances and location similarity use **high + medium** as the best available evidence. **Unknown** units get
  no location score and fall into a third "unknown" group in distance filters, never silently kept or dropped.
* Each unit's confidence goes into the AI's context, so answers can report coverage (e.g. "of the 10 closest,
  3 have high-confidence locations").
* A **strict mode** (high only) is available on request, or as a sensitivity check.

### 2.3 Build Year

In [22]:
profile("Build Year")

Build Year  (1,250 units)
  blank:          16
  non-numeric:     0  {}
  zero:            0
  numeric:      1234


,count,mean,std,min,25%,50%,75%,max
Build Year,1234.0,1995.48,16.55,1939.0,1982.0,1996.0,2008.75,2022.0


In [23]:
# How much does build year vary within an estate? (Would an estate average be a fair fill-in?)
year = pd.to_numeric(raw_portfolio["Build Year"])
spread = year.groupby(raw_portfolio["Industrial Estate Name"]).agg(["count", "min", "max"])
spread = spread[spread["count"] > 1]
print(f"median within-estate range: {(spread['max'] - spread['min']).median():.0f} years")

median within-estate range: 31 years


* Clean: 16 blanks, no text, range 1939–2022.
* Units in the same estate differ by ~30 years (median), so filling blanks with the estate average would
  be a guess. **Leave them missing**: their age score is unavailable.

### 2.4 Car Parking Spaces #

In [24]:
profile("Car Parking Spaces #")

Car Parking Spaces #  (1,250 units)
  blank:           0
  non-numeric:     0  {}
  zero:            0
  numeric:      1250


,count,mean,std,min,25%,50%,75%,max
Car Parking Spaces #,1250.0,30.05,14.24,5.0,17.0,30.5,44.0,50.0


In [25]:
# One value per estate (every unit in an estate has the same figure, section 4.3).
parking = pd.to_numeric(raw_portfolio["Car Parking Spaces #"])
per_estate = parking.groupby(raw_portfolio["Industrial Estate Name"]).first()
px.histogram(x=per_estate, nbins=46, title="Car parking spaces, one value per estate",
             labels={"x": "Car Parking Spaces #"}, height=320).show()

size = pd.to_numeric(raw_portfolio["Size sqm"])
print(f"{len(per_estate)} estates; each value 5-50 used by "
      f"{per_estate.value_counts().min()}-{per_estate.value_counts().max()} estates")
print(f"rank correlation with unit size: {parking.corr(size, method='spearman'):.2f}")

263 estates; each value 5-50 used by 1-11 estates


rank correlation with unit size: -0.15


* Complete, with values 5–50. Every unit in an estate has the same value (section 4.3), so the unit
  histogram is spiky: each spike is one estate repeated (e.g. 97 units at 12 spaces).
* Counted once per estate, the values are spread almost evenly across 5–50 and don't rise with unit size
  (rank correlation −0.15). That looks closer to random than to a real attribute.
* We can't tell whether it's an estate total or one unit's allocation copied across.
* **Excluded from similarity scoring; shown as context only.**

### 2.5 Min. Eaves m and Max. Eaves m

In [26]:
profile("Min. Eaves m")

Min. Eaves m  (1,250 units)
  blank:         437
  non-numeric:    16  {'X': 16}
  zero:          371
  numeric:       797


,count,mean,std,min,25%,50%,75%,max
Min. Eaves m,797.0,3.44,3.8,0.0,0.0,3.4,6.0,18.0


In [27]:
profile("Max. Eaves m")

Max. Eaves m  (1,250 units)
  blank:         442
  non-numeric:   120  {'-': 104, 'X': 16}
  zero:          401
  numeric:       688


,count,mean,std,min,25%,50%,75%,max
Max. Eaves m,688.0,3.52,4.62,0.0,0.0,0.0,6.7,18.0


In [28]:
# Max minus min eaves should never be negative. Zeros mean "not recorded" (see below), so use units
# where both heights are recorded and above zero.
min_eaves = pd.to_numeric(raw_portfolio["Min. Eaves m"], errors="coerce")
max_eaves = pd.to_numeric(raw_portfolio["Max. Eaves m"], errors="coerce")
both = (min_eaves > 0) & (max_eaves > 0)
eaves_gap = (max_eaves - min_eaves)[both]

fig = px.histogram(x=eaves_gap, nbins=60, title="Max eaves minus min eaves (units with both recorded)",
                   labels={"x": "max - min (m)"}, height=320)
fig.add_vline(x=0, line_dash="dash")
fig.show()

print(f"units with both recorded: {both.sum()}")
print(f"  negative (min > max):   {(eaves_gap < 0).sum()}")
print(f"  zero (flat roof):       {(eaves_gap == 0).sum()}")
print(f"  positive:               {(eaves_gap > 0).sum()}")
print(f"min > max where one side is zero: {((min_eaves > max_eaves) & ~both).sum()}")
print(f"eaves over 15 m:                  {(max_eaves > 15).sum()} units")
display(eaves_gap.describe().round(2).to_frame("max - min").T)

units with both recorded: 277
  negative (min > max):   0
  zero (flat roof):       88
  positive:               189
min > max where one side is zero: 49
eaves over 15 m:                  4 units


,count,mean,std,min,25%,50%,75%,max
max - min,277.0,1.59,1.74,0.0,0.0,1.1,2.5,6.2


* Placeholder text: `X` (16 in each) and `-` (104 in max). These mean "unknown", so treat them as **missing**.
* About 30% are **zero**. No warehouse has 0 m eaves, so zero also means "not recorded". Treat as missing.
  Usually both sides are zero (352 units), but sometimes only one is (49 max only, 19 min only); keep the
  side that was recorded.
* **Max minus min is never negative** for the 277 units with both heights recorded: 88 are equal (a flat
  roof, or one figure quoted twice) and 189 have max above min, by up to 6.2 m.
* The 49 units that showed min > max in the raw data are all cases where **max is zero**, i.e. not
  recorded. They aren't swapped values, so nothing needs swapping once zeros are treated as missing.

### 2.6 Doors #

In [29]:
profile("Doors #")

Doors #  (1,250 units)
  blank:         485
  non-numeric:     4  {'-': 3, 'TBC': 1}
  zero:          542
  numeric:       761


,count,mean,std,min,25%,50%,75%,max
Doors #,761.0,4.22,10.35,0.0,0.0,0.0,5.0,170.0


In [30]:
# Do zero doors and zero eaves occur together? If so, zero is a "not recorded" placeholder.
doors = pd.to_numeric(raw_portfolio["Doors #"], errors="coerce")
pd.crosstab(min_eaves == 0, doors == 0, rownames=["eaves = 0"], colnames=["doors = 0"])

doors = 0,False,True
eaves = 0,,
False,700,179
True,8,363


* Placeholder text: `-` (3) and `TBC` (1). Treat as **missing**.
* 43% are zero, and 363 of the 371 zero-eaves units also have zero doors: the same "not recorded"
  pattern. Treat zero as missing.

In [31]:
# Is the door count the same for every unit in an estate? Does it track the unit's size or the estate's?
recorded = doors > 0
size = pd.to_numeric(raw_portfolio["Size sqm"])
estate = raw_portfolio["Industrial Estate Name"]
door_estates = pd.DataFrame({
    "units": estate.groupby(estate).size(),
    "doors": doors[recorded].groupby(estate[recorded]).first(),
    "distinct counts": doors[recorded].groupby(estate[recorded]).nunique(),
    "estate sq ft": size.groupby(estate).sum(),
}).dropna()
multi_doors = door_estates[door_estates["units"] > 1]
print(f"multi-unit estates with doors recorded: {len(multi_doors)}; "
      f"same count on every unit: {(multi_doors['distinct counts'] == 1).sum()}")
print(f"rank correlation of doors with unit size:   {doors[recorded].corr(size[recorded], method='spearman'):.2f}")
print(f"rank correlation of doors with estate size: "
      f"{door_estates['doors'].corr(door_estates['estate sq ft'], method='spearman'):.2f}")
print(f"median across {len(multi_doors)} multi-unit estates of (estate door count / units in estate): "
      f"{(multi_doors['doors'] / multi_doors['units']).median():.1f}")
raw_portfolio.loc[estate == "Ruby Arc 89"].set_index("Property Id")[
    ["Industrial Estate Name", "Unit Name", "Size sqm", "Doors #"]].head()

multi-unit estates with doors recorded: 32; same count on every unit: 30
rank correlation of doors with unit size:   0.04
rank correlation of doors with estate size: 0.60
median across 32 multi-unit estates of (estate door count / units in estate): 1.5


,Industrial Estate Name,Unit Name,Size sqm,Doors #
Property Id,,,,
234,Ruby Arc 89,Unit 1,3705.756781,27
235,Ruby Arc 89,Unit 2,1508.197741,27
236,Ruby Arc 89,Unit 3,1335.671134,27
237,Ruby Arc 89,Unit 4,3810.769446,27
238,Ruby Arc 89,Unit 5,2749.854228,27


* **Estate-level, like parking:** 30 of the 32 multi-unit estates give every unit the same door count. Ruby Arc 89
  shows 27 doors on each of its 21 small units: implausible per unit, but very plausible as an estate total
  (about 1.3 per unit).
* Doors barely relate to the unit's own size (0.04) but clearly track the estate's total size (0.60). So the figure is best read as an **estate total copied onto each unit**.
* **How the 1.5 is calculated:** each multi-unit estate has one door count, so for each estate we divide it by the
  number of units in that estate (Ruby Arc 89: 27 ÷ 21 ≈ 1.3), giving one ratio per estate, then take the median
  of those 32 ratios. It's what each unit would get if the estate total were shared evenly: 1–2 doors per unit is
  realistic, 27 each is not. (Unit counts include units with doors missing; the 2 estates with mixed counts use
  the first unit's value.)

In [32]:
# Single-unit estates (not part of a larger estate): here the door count describes one building.
single = door_estates[door_estates["units"] == 1]
px.histogram(x=single["doors"], nbins=60, title="Doors per building, single-unit estates only",
             labels={"x": "Doors #"}, height=320).show()
print(f"{len(single)} single-unit estates with doors recorded; rank correlation of doors with size: "
      f"{single['doors'].corr(single['estate sq ft'], method='spearman'):.2f}")
display(single["doors"].describe().round(1).to_frame().T)

44 single-unit estates with doors recorded; rank correlation of doors with size: 0.61


,count,mean,std,min,25%,50%,75%,max
doors,44.0,19.8,27.7,2.0,6.0,11.0,24.2,170.0


* 44 single-unit estates have doors recorded: median 11, half between 6 and 24. Most are 2–40, which is a
  realistic range for standalone sheds.
* Here doors **do** rise with the building's size (rank correlation 0.61), unlike the 0.04 across all units.
  So once the copied estate totals are taken out, doors behave like a real physical attribute.
* Three values stand out on the right: 64, 72 and 170. Whether they're plausible depends on the building's
  size, which the next plot checks.

In [33]:
# Doors against size for single-unit estates. The dashed lines show 1 door per 2,000 sq ft (the doubtful
# threshold) and the median of 1 door per ~9,000 sq ft.
property_id = raw_portfolio["Property Id"].groupby(estate).first()  # one unit per estate here
plot_data = single.reset_index()
plot_data["Property Id"] = plot_data["Industrial Estate Name"].map(property_id)
fig = px.scatter(plot_data, x="estate sq ft", y="doors", hover_name="Industrial Estate Name", hover_data=["Property Id"],
                 title="Doors vs size, single-unit estates", labels={"estate sq ft": "Size sq ft", "doors": "Doors #"},
                 height=420)
max_sqft = single["estate sq ft"].max()
fig.add_scatter(x=[0, max_sqft], y=[0, max_sqft / 2000], mode="lines", line_dash="dash", name="1 per 2,000 sq ft")
fig.add_scatter(x=[0, max_sqft], y=[0, max_sqft / 9000], mode="lines", line_dash="dot", name="1 per 9,000 sq ft")
fig.show()

* Most buildings sit between the two lines, around the 1-per-9,000 line: doors grow with size as expected.
* **Too many doors** (above the 2,000 line): Opal Forge 42 (170 doors, 199k sq ft) and Granite Summit 84 (64, 77k)
  stand well clear of the rest. Topaz Mews 48 (72, 342k) is high but could be a cross-dock building.
* **Too few doors**, which the sq ft per door threshold doesn't catch: Granite Oaks 47 (9 doors on 1.2m sq ft,
  the largest unit in the portfolio), Granite Terrace 80 (2 on 86k), and Velvet Summit 5, Ivory Bridge 30 and
  Azure Pavilion 36 (5–6 doors on 180–220k).
* **The odd ones sit together in the file.** Property Ids 1173–1187 contain both the tiny units with too many doors
  (551–5,400 sq ft, 6–10 doors, 10–12.5 m eaves) and the huge units with too few (86k–1.2m sq ft, 2–9 doors,
  5.7–10 m eaves). A 551 sq ft unit with 12 m eaves isn't believable, so in this block the doors and eaves look
  attached to the wrong units rather than mistyped. Treat **doors and eaves in Property Ids 1173–1187 as doubtful**.

In [34]:
# Single-unit estates: sq ft per door.
sqft_per_door = single["estate sq ft"] / single["doors"]
fig = px.histogram(x=sqft_per_door, nbins=40, title="Single-unit estates: sq ft per door",
                   labels={"x": "sq ft per door"}, height=320)
fig.add_vline(x=2000, line_dash="dash", annotation_text="2,000")
fig.show()
print(f"{len(single)} single-unit estates with doors recorded; median {sqft_per_door.median():,.0f} sq ft per door")

44 single-unit estates with doors recorded; median 8,987 sq ft per door


* In **single-unit estates** the count describes one building. The median is about one door per 9,000 sq ft,
  which is realistic for a logistics shed. Most of the large counts are here and are believable, e.g.
  Steel Harbor 78 (34 doors, 495k sq ft) and Topaz Mews 48 (72 doors, 342k sq ft, possibly a cross-dock).

In [35]:
# Flag: fewer than 2,000 sq ft per door is implausibly dense.
single.assign(**{"sq ft per door": sqft_per_door.round(0)})[sqft_per_door < 2000].sort_values("sq ft per door")

,units,doors,distinct counts,estate sq ft,sq ft per door
Industrial Estate Name,,,,,
Crimson Meadow 78,1,10.0,1.0,765.640329,77.0
Quartz Works 36,1,6.0,1.0,551.147072,92.0
Cedar Hub 43,1,8.0,1.0,1755.126266,219.0
Amber Oaks 57,1,6.0,1.0,5396.115333,899.0
Opal Forge 42,1,170.0,1.0,199154.647208,1171.0
Granite Summit 84,1,64.0,1.0,76867.795201,1201.0
Granite Depot 69,1,3.0,1.0,4042.864350,1348.0


* **7 units fall below 2,000 sq ft per door.** Four of them (Amber Oaks 57, Quartz Works 36, Crimson Meadow 78,
  Cedar Hub 43) are tiny units in Property Ids 1173–1187, which section 2.8 finds are doubtful as a block.
* This threshold only catches *too many* doors, and is harsh on small units. The scatter above shows the same
  block also has huge units with *too few* (e.g. Granite Oaks 47, 9 doors on 1.2m sq ft), which it can't flag;
  and Granite Depot 69 (3 doors on 4,043 sq ft) is plausible.
* **Decision:** flag doors as doubtful in two ways:
  * **a general rule:** fewer than 2,000 sq ft per door, where the count is the unit's own and **more than 3
    doors** (a few doors on a small unit is normal, which spares Granite Depot 69). It catches **Opal Forge 42**
    (170 doors on 199k sq ft) and **Granite Summit 84** (64 on 77k), and it also works on new listings: marketed
    listing M4 has 16 doors on 617 sq ft;
  * **by Property Id:** doors (and eaves and yard depth) in **Property Ids 1173–1187**, whatever their values
    (section 2.8), since the threshold can't catch the block's too-few-doors half.
* Use doors as a physical feature only where the count is **the unit's own**: single-unit estates, and the two
  multi-unit estates whose counts differ by unit and rise with size (Crimson Quay 31, Topaz Arc 37). Elsewhere
  it's an estate total; show it as estate context.

### 2.7 Yard Depth m

In [36]:
profile("Yard Depth m")

Yard Depth m  (1,250 units)
  blank:        1216
  non-numeric:    12  {'20-28': 11, '25-33': 1}
  zero:            0
  numeric:        22


,count,mean,std,min,25%,50%,75%,max
Yard Depth m,22.0,46.43,13.26,24.0,36.25,47.0,50.0,82.5


* 97% blank in the portfolio and blank for every marketed listing, plus some ranges (`20-28`).
  There's nothing on the marketed side to compare against.

In [37]:
# Could a blank mean "no yard"? If so, blanks should depend on the building, not on how complete the record is.
yard = raw_portfolio["Yard Depth m"].notna()
size = pd.to_numeric(raw_portfolio["Size sqm"])
eaves_recorded = pd.to_numeric(raw_portfolio["Max. Eaves m"], errors="coerce") > 0
print(f"units with a yard depth: {yard.sum()} in {raw_portfolio.loc[yard, 'Industrial Estate Name'].nunique()} estates")
print(f"max eaves recorded: {eaves_recorded[yard].mean():.0%} of units with a yard depth, "
      f"{eaves_recorded[~yard].mean():.0%} of units without")
print(f"units over 100k sq ft with no yard depth: {(~yard[size > 100_000]).mean():.0%} of {(size > 100_000).sum()}")
block = raw_portfolio["Property Id"].between(1173, 1187)
print(f"yard depths in Property Ids 1173-1187: {(yard & block).sum()}")
raw_portfolio.loc[yard & block].set_index("Property Id")[
    ["Industrial Estate Name", "Size sqm", "Max. Eaves m", "Doors #", "Yard Depth m"]].head(6)

units with a yard depth: 34 in 21 estates
max eaves recorded: 97% of units with a yard depth, 21% of units without
units over 100k sq ft with no yard depth: 86% of 64
yard depths in Property Ids 1173-1187: 11


,Industrial Estate Name,Size sqm,Max. Eaves m,Doors #,Yard Depth m
Property Id,,,,,
1173,Sapphire Harbour 72,1580.023446,12.5,8,82.5
1174,Amber Oaks 57,5396.115333,10,6,40
1175,Quartz Works 36,551.147072,12,6,50
1176,Sable Pavilion 17,1837.395509,10,8,50
1177,Crimson Meadow 78,765.640329,12.5,10,53
1178,Cedar Hub 43,1755.126266,12.5,8,48


* **A blank means "not recorded", not "no yard":**
  * It tracks how complete the record is: units with a yard depth have max eaves recorded 97% of the time,
    against 21% for units without. Yard depth appears where someone filled in the whole spec sheet.
  * 86% of units over 100k sq ft have no yard depth, and a large logistics shed without a service yard isn't
    believable.
  * Where it is recorded it's often copied across an estate, e.g. `20-28` on all 11 units of Ruby Tower 24.
* **Property Ids 1173–1187 again:** 11 of the 34 yard depths are in this block, with the same mismatch as doors and
  eaves, e.g. a 50 m yard on a 551 sq ft unit and 82.5 m on 1,580 sq ft (section 2.8).
* **Decision: drop it**, and don't use "has a yard" either: it would encode "has a fuller record", not a
  property of the building. Show a recorded yard depth as context only.

### 2.8 Do the odd values cluster in the file?

Sections 2.6 and 2.7 kept pointing at Property Ids 1173–1187: tiny units with tall eaves, many doors and deep
yards, and huge units with few doors and low eaves. Can we show that without pointing at those units ourselves?

This isn't a hunt for individual outliers. It asks whether units that don't make physical sense **sit next to
each other** in the file.

**Step 1: score each unit for how far its specs disagree with its size.** Bigger buildings normally have taller
eaves and more doors, so size predicts roughly what the other two should be.
1. *Fit the trend:* a straight line of max eaves against size, with size on a log scale, because eaves rise with
   the order of magnitude of a building (5k → 50k → 500k sq ft), not with every extra square foot.
2. *Measure each unit's miss:* how far its eaves are from the line. A 551 sq ft unit with 12 m eaves is far above it.
3. *Standardise the miss:* divide by the typical miss, measured with the median absolute deviation, which a few
   extreme units can't inflate the way they inflate a standard deviation. This is a robust z-score: 0 is "as
   expected", 3 is "very unusual". The sign is dropped, so too tall and too short both count.
4. *Repeat for doors*, also on a log scale because door counts are heavily skewed.
5. *Average the two* into one **odd score** per unit (or use the one available).

**Step 2: scan the file for a run of high scores.**
1. Put the scored units in Property Id order, slide a window of 8 consecutive units along them, and keep the
   window with the highest average score.
2. *Is that run unusual or just luck?* Shuffle the scores into a random order 2,000 times and redo the scan each
   time. The p-value is the share of shuffles whose best window beats the real one.

Steps 1 and 2 never use the Property Id to decide what's odd; the algorithm only reports where the run of high
scores turns out to be.

**Step 3: remove a known cause, then check direction.**
1. Scan all units. Estate copying (one estate's figure on every unit, with consecutive Ids) is a known cause of
   runs of odd values, so expect it to show up.
2. Scan single-unit estates only, where values describe the building itself and copying can't happen.
3. Compare the eaves-size relationship inside the suspected block with the rest of the portfolio. Shuffled
   attributes would reverse it; random typos would not.

**Why not DBSCAN?** DBSCAN labels points in sparse regions of the data. Here each odd unit is unremarkable on any
single measure (12 m eaves is normal, 551 sq ft is normal); only the combination with size is odd. Copied estate
values also form artificial dense clusters, and most units lack eaves or doors. Scoring "disagreement with size"
targets the actual problem, and every flag comes with a reason (e.g. "eaves 3 robust z-scores above what its size
predicts").

In [38]:
def oddness(y: pd.Series, size: pd.Series) -> pd.Series:
    """How far each value is from the trend with size, in robust standard deviations (sign removed).

    Fits a straight line of y against log10(size), since eaves and doors grow with the scale of a building,
    then divides each unit's distance from the line by the median absolute deviation (robust to outliers).
    """
    x = np.log10(size)
    ok = y.notna() & x.notna()
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    resid = y - (intercept + slope * x)
    return (resid / (1.4826 * (resid - resid.median()).abs().median())).abs()


size = pd.to_numeric(raw_portfolio["Size sqm"])
eaves = pd.to_numeric(raw_portfolio["Max. Eaves m"], errors="coerce").mask(lambda s: s == 0)
door_log = np.log10(doors.where(doors > 0))
odd_score = pd.concat([oddness(eaves, size), oddness(door_log, size)], axis=1).mean(axis=1).dropna()
odd_score.index = raw_portfolio.loc[odd_score.index, "Property Id"]  # label each score by its Property Id

print(f"units scored (eaves or doors recorded): {len(odd_score)}; median score {odd_score.median():.2f}")
odd_score.sort_values(ascending=False).head(10).round(2).to_frame("odd score").join(
    raw_portfolio.set_index("Property Id")[["Industrial Estate Name", "Unit Name", "Size sqm", "Max. Eaves m", "Doors #"]])

units scored (eaves or doors recorded): 346; median score 0.73


,odd score,Industrial Estate Name,Unit Name,Size sqm,Max. Eaves m,Doors #
Property Id,,,,,,
376,2.66,Sapphire Harbour 72,Unit 1,55995.481597,16,0
504,2.57,Teal Harbor 3,Unit 1,28089.701446,15,0
505,2.51,Teal Harbor 3,Unit 2,31153.190732,15,0
966,2.50,Umber Meadow 63,Unit 1,261693.827973,17.5,NaN
1181,2.31,Granite Terrace 80,Unit 1,86050.111175,6,2
503,2.25,Scarlet Arc 73,Unit 1,134578.52641,18,40
506,2.15,Teal Harbor 3,Unit 3,59428.457801,15,0
906,2.14,Topaz Junction 57,Unit 1,13207.41069,6,1
921,2.11,Ruby Tower 24,Unit 1,4995.704312,12,NaN


* The most unusual single units are mostly **estate copying**, not problems with a run of units: Teal Harbor 3 and Marble
  Beacon 76 copy 15 m eaves onto every unit, including small ones, and Ruby Tower 24 copies 12 m onto a 5k sq ft unit.
* Only one unit from Property Ids 1173–1187 makes the top 10 (Granite Terrace 80, 6 m eaves and 2 doors on 86k sq ft).
  On their own, the block's units are odd but not extreme; what stands out is that they sit together.

In [39]:
def scan(score: pd.Series, window: int = 8, shuffles: int = 2000) -> pd.Series:
    """Find the run of `window` consecutive scored units (in Property Id order) with the highest mean score.

    Returns the run's first and last Property Id, its mean score, and the share of shuffled orders whose best
    run scores at least as high (a permutation p-value).
    """
    rolling = score.rolling(window).mean()
    end = score.index.get_loc(rolling.idxmax())
    ids = score.index[end - window + 1 : end + 1]
    rng = np.random.default_rng(0)
    shuffled = [pd.Series(rng.permutation(score.to_numpy())).rolling(window).mean().max() for _ in range(shuffles)]
    return pd.Series({"first Property Id": ids[0], "last Property Id": ids[-1], "mean score": round(rolling.max(), 2),
                      "p (shuffled order)": np.mean(np.array(shuffled) >= rolling.max())})


scan(odd_score)

first Property Id     488.00
last Property Id      507.00
mean score              1.96
p (shuffled order)      0.00
dtype: float64

* Scanning **all units**, the most unusual run is **Property Ids 488–507**, and no shuffled order produces a run that
  unusual (p < 0.001).
* That run is Marble Beacon 76, Scarlet Arc 73 and Teal Harbor 3, so it's the **estate copying** above appearing as
  a run of Property Ids, because an estate's units have consecutive Ids. That's a real finding, but a
  different one: copied estate values, not misaligned units.

In [40]:
# Repeat with single-unit estates only, where eaves and doors describe the building itself.
estate_of = raw_portfolio.set_index("Property Id")["Industrial Estate Name"]
estate_units = estate_of.map(estate_of.value_counts())  # units in each unit's estate, by Property Id
single_score = odd_score[estate_units[odd_score.index] == 1]
print(f"single-unit estates scored: {len(single_score)}")
scan(single_score)

single-unit estates scored: 51


first Property Id     1177.000
last Property Id      1187.000
mean score               1.700
p (shuffled order)       0.027
dtype: float64

* With copying removed (single-unit estates only), the most unusual run is **Property Ids 1177–1187**: the block we
  suspected, found without being told where to look. A run this unusual turns up in about 3% of shuffled orders
  (p ≈ 0.03).
* The evidence is moderate, not strong: only 51 single-unit estates have eaves or doors recorded, the block
  contributes about 10 of them, and the p-value depends on choices (fitting the size trend on single-unit
  estates alone gave p ≈ 0.06). We also went looking for this block after seeing it, so this is a check on a
  suspicion rather than a blind discovery.

In [41]:
# Direction: across the portfolio, taller eaves go with bigger units. Inside the block?
in_block = raw_portfolio["Property Id"].between(1173, 1187)
both = eaves.notna()
print(f"rank correlation of max eaves with size, Property Ids 1173-1187: "
      f"{eaves[both & in_block].corr(size[both & in_block], method='spearman'):.2f}")
print(f"rank correlation of max eaves with size, all other units:       "
      f"{eaves[both & ~in_block].corr(size[both & ~in_block], method='spearman'):.2f}")

points = pd.DataFrame({"Size sq ft": size, "Max. Eaves m": eaves,
                       "group": np.where(in_block, "Property Ids 1173-1187", "other units"),
                       "estate": raw_portfolio["Industrial Estate Name"], "Property Id": raw_portfolio["Property Id"]}).dropna()
px.scatter(points, x="Size sq ft", y="Max. Eaves m", color="group", hover_name="estate", hover_data=["Property Id"],
           height=420, title="Max eaves vs size: Property Ids 1173-1187 highlighted").show()

rank correlation of max eaves with size, Property Ids 1173-1187: -0.67
rank correlation of max eaves with size, all other units:       0.62


* **The direction is the clearest evidence.** Across the portfolio, taller eaves go with bigger units (+0.62).
  In Property Ids 1173–1187 the relationship reverses (−0.67): the small units have tall eaves and the huge ones short
  eaves. Shuffling attributes between units would produce exactly this; random typos would not.
* On the plot, the block's points sit in the top-left (tiny, tall) and bottom-right (huge, low) corners, away
  from the rest.

**Conclusion:** consistent with doors, eaves and yard depth being attached to the wrong units in Property Ids 1173–1187, though
not proven. We can't tell which side is misplaced, the sizes or the specs. Size is the best-supported field (its total
matches Indurent's published figure, 2.1), so flag **doors, eaves and yard depth as doubtful** for these units. Separately, the scan confirms that estate copying is the largest source of oddness in eaves
and doors.

### 2.9 Other suspect units

A sweep of plausibility checks across every field (eaves under 5 m or over 15 m, build years outside 1950–2022,
EPC against build year, very large units, duplicate names or coordinates, gaps in unit numbering) found three
more things worth recording. Duplicates and numbering were clean.

In [42]:
# Granite Mews 59: every field disagrees with the others.
raw_portfolio.loc[raw_portfolio["Industrial Estate Name"] == "Granite Mews 59"].set_index("Property Id")[
    ["Industrial Estate Name", "Unit Name", "Size sqm", "Min. Eaves m", "Max. Eaves m", "Build Year", "EPC Rating"]]

,Industrial Estate Name,Unit Name,Size sqm,Min. Eaves m,Max. Eaves m,Build Year,EPC Rating
Property Id,,,,,,,
259,Granite Mews 59,Unit 1,5449.316133,2.1,2.4,1940,A
260,Granite Mews 59,Unit 2,2890.247917,2.1,2.4,1940,D
261,Granite Mews 59,Unit 3,5619.207198,2.1,2.4,1939,A
262,Granite Mews 59,Unit 4,8514.75494,2.1,2.4,1949,D
263,Granite Mews 59,Unit 5,2291.938724,2.1,2.4,1953,A


* **Granite Mews 59 (Property Ids 259–263)** doesn't hang together:
  * eaves of 2.1–2.4 m on every unit: ceiling height for a garage, not a warehouse (the next lowest max eaves in
    the portfolio is 5.2 m);
  * build years 1939–1953, four of them the only units in the portfolio built before 1950;
  * EPC A on three of them, i.e. 1940s buildings with the top energy rating.
* Each value could be explained on its own (a refurbished pre-war building, an unusual low-roofed unit). It sits
  next to Granite Depot 69 (Property Id 264). **Flag its eaves as doubtful**: 2.1 m isn't a warehouse.
* **Its build years are kept.** A 1940s industrial building is plausible, and the EPC A ratings that seem to
  contradict them can't be relied on: section 2.9's EPC check below finds the ratings unreliable throughout, and
  doesn't count pre-1970 A ratings as evidence. Size and location stay usable too.

In [43]:
# EPC against build year: newer buildings should mostly rate A or B.
year = pd.to_numeric(raw_portfolio["Build Year"], errors="coerce")
epc = raw_portfolio["EPC Rating"].where(raw_portfolio["EPC Rating"].isin(list("ABCDEFG")))
epc_rank = epc.map({rating: n for n, rating in enumerate("ABCDEFG", start=1)})  # 1 = A (best)
print(f"rank correlation of EPC (1 = A) with build year: {epc_rank.corr(year, method='spearman'):.2f}")
print(f"EPC shares among rated units: {(epc.value_counts(normalize=True) * 100).round(0).to_dict()}")
pd.crosstab(pd.cut(year, [1930, 1970, 1990, 2010, 2015, 2023]), epc, margins=True, margins_name="All")

rank correlation of EPC (1 = A) with build year: 0.16
EPC shares among rated units: {'A': 30.0, 'C': 25.0, 'D': 22.0, 'E': 15.0, 'B': 7.0}


EPC Rating,A,B,C,D,E,All
Build Year,,,,,,
"(1930, 1970]",13,4,23,19,2,61
"(1970, 1990]",108,22,79,67,26,302
"(1990, 2010]",108,23,93,77,42,343
"(2010, 2015]",13,5,15,14,9,56
"(2015, 2023]",33,7,21,30,55,146
All,275,61,231,207,134,908


* **EPC doesn't behave like real ratings.** New buildings must meet current building regulations, so they
  should mostly rate A or B. Here the most common rating for buildings from 2016 onwards is **E** (55 of 146),
  and across the portfolio newer buildings have slightly *worse* ratings (rank correlation +0.16, where higher
  means worse). A is the most common rating overall (30%).
* The pre-1970 A ratings (13) are *not* counted as evidence: Indurent invests in its assets, so refurbished old
  buildings rated A are plausible.
* Coverage is also patchy: 26% of the portfolio is unrated (plus 7 `False`), and 2 of the 5 marketed listings
  have no rating.
* EPC isn't one of the brief's similarity dimensions (physical, location, age), but it matters to acquisitions as
  a **risk**: under the minimum energy efficiency rules a property generally can't be let below E.
* **Decision: exclude EPC from the similarity scores; show it as context, with a caveat that the portfolio's
  ratings look unreliable.**

In [44]:
# Very large units: possible, but rare even for big UK logistics sheds.
size = pd.to_numeric(raw_portfolio["Size sqm"])
estate_units = raw_portfolio["Industrial Estate Name"].map(raw_portfolio["Industrial Estate Name"].value_counts())
raw_portfolio.assign(**{"units in estate": estate_units})[size > 600_000].set_index("Property Id")[
    ["Industrial Estate Name", "Unit Name", "units in estate", "Size sqm", "Max. Eaves m", "Doors #", "Build Year"]]

,Industrial Estate Name,Unit Name,units in estate,Size sqm,Max. Eaves m,Doors #,Build Year
Property Id,,,,,,,
348,Marble Works 29,Unit 1,1,887038.323443,10.8,31,1987
377,Sapphire Harbour 72,Unit 2,3,757853.495896,16,0,2022
1145,Crimson Vista 3,Unit 1,1,616800.576607,NaN,NaN,1994
1170,Stone Junction 93,Unit 1,1,779137.200639,NaN,NaN,1997
1179,Granite Oaks 47,Unit 1,1,1195450.570446,10,9,1987


* Five units are over 600k sq ft. Even large UK logistics sheds are mostly under 500k, so these are rare but
  possible. One (Granite Oaks 47, 1.2m sq ft) is already in the doubtful block (2.8).
* Sapphire Harbour 72 copies 16 m eaves onto both its 56k and 758k sq ft units: estate copying again.
* **Decision:** don't flag these as wrong; note them as **very large** so that answers involving them can say so.

## 3. Converting to numbers

Applying the decisions above gives typed `portfolio` and `marketed` tables for the rest of the notebook:

* placeholder text → missing; ranges → midpoint
* zero eaves and zero doors → missing
* `Size sqm` renamed to `Size sq ft`: the values are square feet (section 2.1)

In [45]:
NUMERIC_COLS = ["Latitude", "Longitude", "Car Parking Spaces #", "Size sqm", "Build Year",
                "Yard Depth m", "Min. Eaves m", "Max. Eaves m", "Doors #"]


def parse_number(value: object) -> float:
    """Numbers pass through; 'a-b' ranges become their midpoint; anything else becomes NaN."""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    text = str(value).strip()
    try:
        low, high = (float(p) for p in text.split("-"))
        return (low + high) / 2
    except ValueError:
        return np.nan


def to_typed(raw: pd.DataFrame) -> pd.DataFrame:
    """Apply the section 2 decisions to a raw (object) frame."""
    df = raw.copy()
    for col in NUMERIC_COLS:
        df[col] = df[col].map(parse_number)
    for col in ["Min. Eaves m", "Max. Eaves m", "Doors #"]:
        df[col] = df[col].mask(df[col] == 0)
    df["EPC Rating"] = df["EPC Rating"].where(df["EPC Rating"].isin(list("ABCDEFG")))
    return df.rename(columns={"Size sqm": "Size sq ft"})  # values are sq ft despite the name (2.1)


portfolio = to_typed(raw_portfolio)
marketed = to_typed(raw_marketed)

# Missing share per column after conversion.
pd.DataFrame({
    "portfolio missing %": portfolio.isna().mean() * 100,
    "marketed missing %": marketed.isna().mean() * 100,
}).round(0)

,portfolio missing %,marketed missing %
Property Id,0.0,100.0
Industrial Estate Name,0.0,0.0
Unit Name,0.0,0.0
Region,0.0,20.0
Latitude,6.0,20.0
Longitude,6.0,20.0
Car Parking Spaces #,0.0,0.0
Size sq ft,0.0,0.0
Build Year,1.0,20.0
Yard Depth m,97.0,100.0


After cleaning, **eaves and doors are mostly missing** (66–82%), so for most units physical
similarity will rest on size and parking. The matcher should report how many features each
comparison actually used.

## 4. Text variables

### 4.1 Region

In [46]:
raw_portfolio["Region"].value_counts(dropna=False)

Region
South East         378
Midlands           301
North West         165
London             144
South West         134
Yorks. & Humber     50
Scotland            38
North East          16
Wales               13
East of England     11
Name: count, dtype: int64

### 4.2 EPC Rating

In [47]:
raw_portfolio["EPC Rating"].value_counts(dropna=False)

EPC Rating
NaN      319
A        276
C        234
D        207
E        142
B         65
False      7
Name: count, dtype: int64

* EPC has 319 blanks and 7 `False` values (an Excel TRUE/FALSE cell, probably an export glitch). Both are
  treated as missing. Only A–E appear.
* It isn't one of the brief's three dimensions, but it matters commercially. It's an optional extra.

### 4.3 Which values belong to the estate rather than the unit?

In [48]:
# For estates with more than one unit: how often is each column the same for every unit?
multi = portfolio.groupby("Industrial Estate Name").filter(lambda g: len(g) > 1)
by_estate = multi.groupby("Industrial Estate Name")
cols = ["Region", "Car Parking Spaces #", "Min. Eaves m", "Max. Eaves m", "Doors #",
        "EPC Rating", "Build Year", "Size sq ft"]
pd.Series({c: (by_estate[c].nunique(dropna=False) == 1).mean() for c in cols},
          name="share of estates where every unit has the same value").round(2).to_frame()

,share of estates where every unit has the same value
Region,0.93
Car Parking Spaces #,1.00
Min. Eaves m,0.95
Max. Eaves m,0.94
Doors #,0.96
EPC Rating,0.52
Build Year,0.01
Size sq ft,0.00


Region, parking, eaves and doors are the same across almost every estate. They were recorded once per
estate and copied to each unit. A 64-unit estate could therefore fill a top-10 result on its own. One option is to **cap units per
estate** in results, for when a user wants a spread across the portfolio (the prompt 1 answer in `03_answers` keeps the literal top 10 by
default, and offers the cap on request).

## 5. Distance to major cities

Distances only mean something where we trust the coordinates, so this section uses the location
confidence flag from 2.2: **high** and **medium** units get a distance, **unknown** units (no coordinates,
or 1 km+ offshore) don't. A distance filter such as "within 10 miles of a major city" then has three
answers rather than two: within, beyond, or unknown.

In [49]:
# Major cities, used for "distance to nearest major city" (prompt 3 in the brief).
MAJOR_CITIES = pd.DataFrame(
    [
        ("London", 51.5074, -0.1278), ("Birmingham", 52.4862, -1.8904),
        ("Manchester", 53.4808, -2.2426), ("Leeds", 53.8008, -1.5491),
        ("Glasgow", 55.8642, -4.2518), ("Liverpool", 53.4084, -2.9916),
        ("Newcastle", 54.9783, -1.6178), ("Sheffield", 53.3811, -1.4701),
        ("Bristol", 51.4545, -2.5879), ("Edinburgh", 55.9533, -3.1883),
        ("Leicester", 52.6369, -1.1398), ("Nottingham", 52.9548, -1.1581),
        ("Cardiff", 51.4816, -3.1791), ("Southampton", 50.9097, -1.4044),
        ("Coventry", 52.4068, -1.5197), ("Bradford", 53.7960, -1.7594),
    ],
    columns=["city", "lat", "lon"],
)


def haversine_miles(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles (works on arrays)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 3958.8 * np.arcsin(np.sqrt(a))


def nearest_city(df: pd.DataFrame) -> pd.DataFrame:
    """Nearest major city and its distance in miles, for each row (NaN where coordinates are missing)."""
    located = df.dropna(subset=["Latitude", "Longitude"])
    dist = pd.DataFrame({
        c.city: haversine_miles(located["Latitude"], located["Longitude"], c.lat, c.lon)
        for c in MAJOR_CITIES.itertuples()
    })
    result = pd.DataFrame({"nearest_city": dist.idxmin(axis=1), "miles_to_city": dist.min(axis=1)})
    return result.reindex(df.index)


# Carry the 2.2 flag onto the typed table (same rows, same order).
portfolio["location_confidence"] = location_confidence

# Only measure distances for units whose location we trust.
usable = portfolio["location_confidence"] != "unknown"
portfolio[["nearest_city", "miles_to_city"]] = nearest_city(portfolio[["Latitude", "Longitude"]].where(usable))

In [50]:
# Same confidence rule for the 5 marketed listings (2.2).
listed = marketed.dropna(subset=["Latitude", "Longitude"])
listed = gpd.GeoDataFrame(
    listed, geometry=gpd.points_from_xy(listed["Longitude"], listed["Latitude"]), crs=4326
).to_crs(27700)
km_offshore = listed.distance(uk_land) / 1000
km_to_region = pd.Series(
    [point.distance(region_shape[label]) / 1000 if label in region_shape.index else np.nan  # Chancery Depot has no region
     for point, label in zip(listed.geometry, listed["Region"])],
    index=listed.index,
)

marketed["location_confidence"] = "unknown"
marketed.loc[listed.index[km_offshore < 1], "location_confidence"] = "medium"
marketed.loc[listed.index[km_to_region <= 1], "location_confidence"] = "high"

marketed[["nearest_city", "miles_to_city"]] = nearest_city(
    marketed[["Latitude", "Longitude"]].where(marketed["location_confidence"] != "unknown")
)
marketed[["Industrial Estate Name", "Unit Name", "Region", "location_confidence", "nearest_city", "miles_to_city"]].round(1)

,Industrial Estate Name,Unit Name,Region,location_confidence,nearest_city,miles_to_city
0,Cherry Lane,Unit 10,North West,medium,Cardiff,72.8
1,Tech Hub,Unit 8,South East,unknown,NaN,NaN
2,Spitfire Park,Unit 42,Midlands,high,Manchester,14.0
3,Stable Lane,Unit 14,South East,medium,London,40.6
4,Chancery Depot,Unit 2,NaN,unknown,NaN,NaN


Two of the five listings get no distance: **Tech Hub** has no coordinates, and **Chancery Depot** is offshore.
Only **Spitfire Park** is high confidence. **Cherry Lane** and **Stable Lane** are medium, because their
coordinates sit outside their labelled regions (mid-Wales and Bedfordshire rather than the North West and South East).

In [51]:
# How many units are within a given distance of a major city: one line per confidence level, and all together.
located = portfolio.dropna(subset=["miles_to_city"])
lines = pd.concat([located.assign(group=located["location_confidence"]), located.assign(group="all located units")])
fig = px.ecdf(lines, x="miles_to_city", color="group", ecdfnorm=None, height=450,
              color_discrete_map={"all located units": "#444444", "medium": "#eb6834", "high": "#2a78d6"},
              labels={"miles_to_city": "Miles to the nearest major city", "group": ""},
              title="Portfolio units within a given distance of a major city")
fig.update_yaxes(title="Units within this distance")
within_10 = (located["miles_to_city"] <= 10).sum()
fig.add_vline(x=10, line_dash="dash",
              annotation_text=f"10 miles: {within_10} units", annotation_position="top right")
fig.show()


* Read each line as "how many units are within this many miles of a major city". At **10 miles only 50 of the
  892 located units qualify**; the count passes 200 at 20 miles and 400 at 30, and half of all located units are
  more than about 31 miles from a city.
* So the brief's 10-mile radius is strict: it keeps about 6% of the units it can check. The radius is a parameter
  of `filter_rows`, so a wider one is a single change in the request.


In [52]:
# The 10-mile filter from prompt 3, as a three-way answer.
ten_miles = pd.Series("beyond 10 miles", index=portfolio.index)
ten_miles[portfolio["miles_to_city"] <= 10] = "within 10 miles"
ten_miles[~usable] = "unknown location"

split = pd.crosstab(ten_miles, portfolio["location_confidence"], margins=True, margins_name="All")
split.index.name = "10-mile filter"
split.loc[["within 10 miles", "beyond 10 miles", "unknown location", "All"]]

location_confidence,high,medium,unknown,All
10-mile filter,,,,
within 10 miles,9,41,0,50
beyond 10 miles,231,611,0,842
unknown location,0,0,358,358
All,240,652,358,1250


* **50 units are within 10 miles** of one of the 16 cities, **842 are beyond**, and **358 are unknown**.
  The unknowns are 29% of the portfolio, so an answer to prompt 3 must say how many units it couldn't check.
* The within-10-miles count didn't change from the unfiltered version: none of the offshore points happened to
  fall near a city. Dropping them still matters, because otherwise an offshore unit could be reported as
  "beyond 10 miles" when its true position is unknown.
* Of the 50 within, only 9 are high confidence; 41 are medium (on land, but region label disagrees). A strict
  mode that keeps only high would leave 9.
* The city list and the 10-mile radius are parameters, and the answer should state them.

## 6. The marketed listings

In [53]:
def percentile(value: float, reference: pd.Series) -> float:
    """Share of portfolio values below `value`, as a percentage (NaN if value missing)."""
    return np.nan if pd.isna(value) else (reference.dropna() < value).mean() * 100


compare = ["Size sq ft", "Max. Eaves m", "Doors #", "Car Parking Spaces #", "Build Year", "miles_to_city"]
summary = marketed[["Industrial Estate Name", "Unit Name", "Region", "location_confidence", "nearest_city"]].copy()
for col in compare:
    summary[f"{col} pctile"] = marketed[col].map(lambda v, c=col: percentile(v, portfolio[c])).round(0)
summary["missing fields"] = marketed[compare[:-1] + ["Latitude"]].isna().sum(axis=1)
summary

,Industrial Estate Name,Unit Name,Region,location_confidence,nearest_city,Size sq ft pctile,Max. Eaves m pctile,Doors # pctile,Car Parking Spaces # pctile,Build Year pctile,miles_to_city pctile,missing fields
0,Cherry Lane,Unit 10,North West,medium,Cardiff,79.0,NaN,NaN,48.0,15.0,97.0,2
1,Tech Hub,Unit 8,South East,unknown,NaN,6.0,NaN,NaN,40.0,NaN,NaN,4
2,Spitfire Park,Unit 42,Midlands,high,Manchester,27.0,NaN,NaN,91.0,13.0,17.0,2
3,Stable Lane,Unit 14,South East,medium,London,2.0,41.0,64.0,85.0,58.0,68.0,0
4,Chancery Depot,Unit 2,NaN,unknown,NaN,78.0,8.0,2.0,33.0,2.0,NaN,0


* **Stable Lane / Unit 14**: the only fully populated listing; very small.
* **Chancery Depot / Unit 2**: coordinates ~27 km offshore in Cardigan Bay (coastline check), and no region. Its location score is unavailable.
* **Tech Hub / Unit 8**: no coordinates or build year. Only a physical comparison is possible.
* **Cherry Lane / Unit 10**: labelled North West but plots in mid-Wales.
* **Spitfire Park / Unit 42**: plausible location (near Manchester), but no eaves or doors.

## 7. Decisions for the pipeline

| Variable | Issue | Decision |
|---|---|---|
| Size sqm | Total (30.6m) matches Indurent's stated 28–40m **sq ft**; eaves and estate sizes agree | Rename to `Size sq ft` (values unchanged); flag the 10 units < 500 sq ft; note the 5 units > 600k sq ft as very large (2.9) |
| Latitude / Longitude | 80 missing (no sibling to borrow from); 278 units 1 km+ offshore (coastline check) | Location confidence flag: high 240, medium 652, unknown 358. Distances and location score for high + medium only; distance filters answer within / beyond / unknown |
| Build Year | 16 missing; varies ~30 years within an estate | Leave missing; age score unavailable |
| Car Parking | Same for every unit in an estate; per estate, spread evenly 5–50 and unrelated to size | Exclude from scoring; show as context only |
| Eaves (min/max) | `X`, `-` placeholders; ~30% zeros; the 49 raw min > max are all max = 0 | Placeholders and zeros → missing (keep the recorded side); no swaps needed |
| Doors # | `-`, `TBC`; 43% zeros (alongside zero eaves); estate total copied onto each unit in 30 of 32 multi-unit estates; implausible values in Property Ids 1173–1187 (2.8) and 2 others | Placeholders and zeros → missing; physical feature for single-unit estates only, context elsewhere; physical feature only where the count is the unit's own; flag counts over 3 doors with < 2,000 sq ft per door (catches Opal Forge 42, Granite Summit 84, listing M4), plus doors in Property Ids 1173–1187 |
| Yard Depth m | 97% missing (none for marketed); blank means not recorded, not "no yard" | Drop; don't derive "has a yard"; context only |
| Region | Only 26% of valid locations are inside their labelled region (boundary check) | Don't use as a feature; derive region from coordinates when needed |
| EPC Rating | 26% missing, 7 `False`; doesn't fit build year (post-2015 builds mostly E) (2.9) | Exclude from similarity scores; show as context, with a caveat that the portfolio's ratings look unreliable |
| Property Ids 1173–1187 | Doors, eaves and yard depth contradict size (tiny units with tall eaves and many doors, huge units with few); found by an order-blind scan (2.8) | Flag doors, eaves and yard depth for these units as doubtful, where recorded; keep size |
| Granite Mews 59 | Eaves 2.1–2.4 m, build years 1939–1953, EPC A on 1940s buildings (2.9) | Flag eaves (Property Ids 259–263) as doubtful; build year, size and location stay usable (a 1940s build is plausible; EPC can't be relied on) |
| Estate structure | Eaves/doors/parking copied across an estate's units | Option: cap units per estate in results when a spread is wanted (off by default, `03_answers`, prompt 1) |

**For the matcher:** score **physical**, **location** and **age** separately; compare only the features
both properties have (don't invent values); and pass the data-quality flags to the LLM so it can caveat
its answers.